In [0]:
# Generate the date range
from pyspark.sql.functions import *

date_df = spark.sql("""
SELECT explode(
        sequence(
            to_date('2020-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )
    ) AS full_date
""")


# Create date attributes
dim_date = date_df \
.withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int")) \
.withColumn("year", year("full_date")) \
.withColumn("month", month("full_date")) \
.withColumn("day", dayofmonth("full_date")) \
.withColumn("week", weekofyear("full_date")) \
.withColumn("quarter", quarter("full_date")) \
.withColumn("day_of_week", date_format("full_date", "EEEE"))


# Save the Gold table
dim_date.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.dim_date")




In [0]:
# Verify
spark.sql("SELECT * FROM gold.dim_date LIMIT 100").show()

In [0]:

orders = spark.read.format("delta") \
    .load("/Volumes/ecom/storage/project/silver/orders/")

In [0]:
from pyspark.sql.functions import date_format, col

# Add date_key column
orders = orders.withColumn(
    "date_key",
    date_format(col("order_date"), "yyyyMMdd").cast("int")
)

# Select fact columns
fact_orders = orders.select(
    "order_id",
    "customer_id",
    "product_id",
    "date_key",
    "quantity",
    "total_amount"
)

# Write as Delta table partitioned by date_key
fact_orders.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("date_key") \
    .saveAsTable("gold.fact_orders")

In [0]:
%sql
-- Optimize
OPTIMIZE gold.fact_orders;

-- Z-Order
OPTIMIZE gold.fact_orders
ZORDER BY (customer_id, product_id);

-- Vacuum the table
VACUUM gold.fact_orders RETAIN 168 HOURS;
VACUUM gold.dim_customers RETAIN 168 HOURS;
VACUUM gold.dim_products RETAIN 168 HOURS;


In [0]:
%sql
--Collects column statistics so the query engine can choose the best plan:
-- Collect statistics for fact_orders
ANALYZE TABLE gold.fact_orders COMPUTE STATISTICS;

-- For dimensions
ANALYZE TABLE gold.dim_customers COMPUTE STATISTICS;
ANALYZE TABLE gold.dim_products COMPUTE STATISTICS;
ANALYZE TABLE gold.dim_date COMPUTE STATISTICS;